### Just loading data from bigquery (gold layer) and turning into DataFrame for EDA

In [ ]:
# Set up 
from google.cloud import bigquery
import pandas as pd
import numpy as np

from dotenv import load_dotenv
import os

load_dotenv()

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
client = bigquery.Client(project=PROJECT_ID)

In [6]:
# Query to pull training table from gold 
# Only rows where target exists (excludes last 12 months)
query = f"""
SELECT
    city,
    state,
    tier,
    month,

    -- Price features
    zhvi_yoy_smooth,
    zhvi_mom_3m,
    zhvi_volatility_6m,

    -- Rent features
    zori_yoy_smooth,
    zori_mom_3m,
    zori_volatility_6m,

    -- Labor features
    unemployment_rate,
    unemployment_3m_delta,
    jobs_3m_pct,
    jobs_yoy_pct,
    wages_yoy_pct,

    -- Supply features
    permits_yoy_pct,

    -- National macro features
    mortgage_rate_3m_delta,
    mortgage_rate_12m_delta,
    mortgage_rate_volatility_6m,
    cpi_yoy_pct,

    -- Affordability features
    price_to_income_ratio,
    mortgage_to_income_ratio,
    rent_to_income_ratio,

    -- Target variable — 12 month forward HPA
    hpa_12m_forward

FROM `{PROJECT_ID}.housing_gold.gold_market_features`
WHERE hpa_12m_forward IS NOT NULL   -- excludes last 12 months (no future yet)
AND zhvi_yoy_smooth IS NOT NULL     -- excludes first 12 months (no lag yet)
AND month >= '2019-01-01'           -- 12m lag warmup complete
ORDER BY city, month
"""

In [9]:
df = client.query(query).to_dataframe(create_bqstorage_client=False)
df

,city,state,tier,month,zhvi_yoy_smooth,zhvi_mom_3m,zhvi_volatility_6m,zori_yoy_smooth,zori_mom_3m,zori_volatility_6m,...,wages_yoy_pct,permits_yoy_pct,mortgage_rate_3m_delta,mortgage_rate_12m_delta,mortgage_rate_volatility_6m,cpi_yoy_pct,price_to_income_ratio,mortgage_to_income_ratio,rent_to_income_ratio,hpa_12m_forward
0,atlanta,GA,2,2019-01-01,0.0865,0.0068,0.0004,0.0664,0.0063,0.0023,...,0.0311,-0.1926,-0.39,0.36,0.1706,0.0149,2.85,0.1386,0.1816,0.0540
1,atlanta,GA,2,2019-02-01,0.0851,0.0068,0.0004,0.0659,0.0050,0.0023,...,0.0202,0.1178,-0.53,0.01,0.1709,0.0152,2.86,0.1378,0.1823,0.0551
2,atlanta,GA,2,2019-03-01,0.0834,0.0062,0.0006,0.0654,0.0040,0.0021,...,0.0208,-0.2169,-0.34,-0.05,0.1560,0.0188,2.88,0.1386,0.1830,0.0565
3,atlanta,GA,2,2019-04-01,0.0803,0.0053,0.0011,0.0644,0.0042,0.0018,...,0.0008,-0.0069,-0.31,-0.38,0.1164,0.0200,2.89,0.1357,0.1839,0.0568
4,atlanta,GA,2,2019-05-01,0.0769,0.0043,0.0015,0.0635,0.0043,0.0013,...,0.0119,-0.2239,-0.27,-0.52,0.0946,0.0180,2.90,0.1352,0.1846,0.0550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2549,washington_dc,VA,1,2024-09-01,0.0369,0.0015,0.0028,0.0485,0.0037,0.0010,...,0.0323,-0.2384,-0.64,-0.96,0.2396,0.0243,4.67,0.2791,0.2303,0.0089
2550,washington_dc,VA,1,2024-10-01,0.0358,0.0024,0.0020,0.0473,0.0030,0.0012,...,0.0116,-0.1943,-0.23,-1.07,0.2657,0.0258,4.69,0.2911,0.2309,0.0062
2551,washington_dc,VA,1,2024-11-01,0.0371,0.0034,0.0013,0.0456,0.0028,0.0012,...,0.0408,0.2623,0.11,-0.92,0.2737,0.0272,4.71,0.2959,0.2316,0.0038
2552,washington_dc,VA,1,2024-12-01,0.0396,0.0038,0.0014,0.0439,0.0027,0.0008,...,0.0549,-0.1264,0.50,-0.18,0.2617,0.0287,4.73,0.2973,0.2322,0.0018


In [12]:
print(f"Rows: {len(df):,}")
print(f"Cities: {df['city'].nunique()}")
print(f"Date range: {df['month'].min()} → {df['month'].max()}")
df.head()

Rows: 2,554
Cities: 35
Date range: 2019-01-01 → 2025-01-01


,city,state,tier,month,zhvi_yoy_smooth,zhvi_mom_3m,zhvi_volatility_6m,zori_yoy_smooth,zori_mom_3m,zori_volatility_6m,...,wages_yoy_pct,permits_yoy_pct,mortgage_rate_3m_delta,mortgage_rate_12m_delta,mortgage_rate_volatility_6m,cpi_yoy_pct,price_to_income_ratio,mortgage_to_income_ratio,rent_to_income_ratio,hpa_12m_forward
0,atlanta,GA,2,2019-01-01,0.0865,0.0068,0.0004,0.0664,0.0063,0.0023,...,0.0311,-0.1926,-0.39,0.36,0.1706,0.0149,2.85,0.1386,0.1816,0.0540
1,atlanta,GA,2,2019-02-01,0.0851,0.0068,0.0004,0.0659,0.0050,0.0023,...,0.0202,0.1178,-0.53,0.01,0.1709,0.0152,2.86,0.1378,0.1823,0.0551
2,atlanta,GA,2,2019-03-01,0.0834,0.0062,0.0006,0.0654,0.0040,0.0021,...,0.0208,-0.2169,-0.34,-0.05,0.1560,0.0188,2.88,0.1386,0.1830,0.0565
3,atlanta,GA,2,2019-04-01,0.0803,0.0053,0.0011,0.0644,0.0042,0.0018,...,0.0008,-0.0069,-0.31,-0.38,0.1164,0.0200,2.89,0.1357,0.1839,0.0568
4,atlanta,GA,2,2019-05-01,0.0769,0.0043,0.0015,0.0635,0.0043,0.0013,...,0.0119,-0.2239,-0.27,-0.52,0.0946,0.0180,2.90,0.1352,0.1846,0.0550


In [14]:
df.to_parquet("../outputs/training_data.parquet", index=False)
print("Saved to outputs/training_data.parquet")

Saved to outputs/training_data.parquet
